# 04 - Clustering No Supervisado y Caracterización de Arquetipos
**Tesis de Maestría en Ciencia de Datos - Universidad Austral**  
*Autor: Nicolás Palavecino | Tutor: Ezequiel Nuske*

---

## 1. Fundamentos de Clustering No Supervisado
Para agrupar a los pilotos sin imponer etiquetas humanas preconcebidas, aplicamos y comparamos múltiples algoritmos:
- **K-Means**: Particionamiento por minimización de inercia intra-clúster.
- **Clustering Jerárquico Aglomerativo**: Agrupamiento jerárquico ascendente con criterio de enlace *Ward*.
- **Modelos de Mezclas Gaussianas (GMM)**: Asignación probabilística (*soft clustering*).
- **Métricas de Selección de $k$**: Método del Codo, Coeficiente de Silueta (*Silhouette Score*), Índice Davies-Bouldin y Calinski-Harabasz.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from f1_style_clustering.data.loader import load_session, enable_cache
from f1_style_clustering.features.zoi import extract_circuit_zones
from f1_style_clustering.features.aggregators import build_session_feature_matrix
from f1_style_clustering.features.preprocessor import normalize_relative_to_mean, standardize_features
from f1_style_clustering.models.dimensionality import perform_pca
from f1_style_clustering.models.clustering import fit_kmeans, fit_agglomerative, fit_gmm
from f1_style_clustering.models.evaluation import find_optimal_k, evaluate_clustering
from f1_style_clustering.analysis.archetypes import characterize_clusters, assign_archetype_names, get_driver_archetype_summary
from f1_style_clustering.visualization.cluster_plots import plot_pca_clusters, plot_optimal_k_evaluation
from f1_style_clustering.visualization.radar_charts import plot_cluster_radar_chart

enable_cache('cache')
session = load_session(year=2025, grand_prix='Italian Grand Prix', session_type='R')
analysis_zones = extract_circuit_zones(session)
df_raw = build_session_feature_matrix(session, analysis_zones=analysis_zones)
df_norm, df_meta = normalize_relative_to_mean(df_raw)
df_standardized, _, _ = standardize_features(df_norm)
df_pca, _, pca_summary = perform_pca(df_standardized, n_components=2)


## 2. Determinación Objetiva del Número Óptimo de Clústeres ($k$)

In [ ]:
# Evaluación en rango k in [2, 6]
df_eval = find_optimal_k(df_standardized, k_min=2, k_max=6)
display(df_eval)

fig_k = plot_optimal_k_evaluation(df_eval)
plt.show()


## 3. Ajuste de Modelos de Clustering y Visualización en Espacio PCA

In [ ]:
# Ajustar K-Means con k=3
K_OPTIMAL = 3
cluster_labels, km_model = fit_kmeans(df_standardized, n_clusters=K_OPTIMAL)
metrics = evaluate_clustering(df_standardized, cluster_labels, km_model)

print(f"Silhouette Score: {metrics.silhouette:.3f} | Davies-Bouldin: {metrics.davies_bouldin:.3f}")

# Caracterización de arquetipos
profiles = characterize_clusters(df_standardized, cluster_labels)
archetype_names = assign_archetype_names(profiles)
df_summary = get_driver_archetype_summary(df_standardized, cluster_labels, df_meta)

display(df_summary)

# Visualización PCA con clústeres
fig_pca_clusters = plot_pca_clusters(df_pca, cluster_labels, pca_summary, archetype_names=archetype_names)
plt.show()


## 4. Perfilado de Arquetipos mediante Gráficos de Radar

In [ ]:
# Gráfico de radar multidimensional
radar_features = [c for c in df_standardized.columns if 'ApexSpeed' in c or 'MaxDecel' in c or 'ThrottleRate' in c][:6]
fig_radar = plot_cluster_radar_chart(profiles, features=radar_features, archetype_names=archetype_names)
plt.show()
